# 예스24 리뷰 크롤링

## 절차
1. 전체리뷰
    - 예스24 메인 접속
    - ISBN 검색 → 첫 번째 도서 상세페이지로 이동
    - '전체리뷰' 탭 클릭 후 스크롤
    - 리뷰 섹션에서 닉네임, 별점, 본문 수집

2. 한줄평 리뷰
    - 전체리뷰와 같은 방식으로 하되 한줄평 영역이 보이도록 스크롤 후 수집

## 1. 전체리뷰

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
from bs4 import BeautifulSoup

def setup_driver():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_experimental_option("detach", True)
    options.add_argument("--disable-blink-features=AutomationControlled")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


def crawl_yes24_reviews(isbn_list):
    all_reviews = []
    driver = setup_driver()

    for idx, isbn in enumerate(isbn_list):
        print(f"\n [{isbn}] ({idx+1}/{len(isbn_list)}) 수집 시작")
        try:
            driver.get("https://www.yes24.com/")
            time.sleep(2)

            search_box = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "span.schIpt input"))
            )
            search_box.clear()
            search_box.send_keys(isbn)

            search_button = driver.find_element(By.CSS_SELECTOR, "button[title='검색']")
            search_button.click()
            time.sleep(2)

            try:
                detail_link = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "em.img_bdr"))
                )
                detail_link.click()
            except:
                print(f"[{isbn}] 상세페이지 이동 실패")
                continue

            time.sleep(3)
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

            try:
                total_tab = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.XPATH, '//li[@class="firstCol on" or @class="firstCol"]//a[contains(., "전체 리뷰")]'))
                )
                driver.execute_script("arguments[0].click();", total_tab)
                print(f"[{isbn}] 전체 리뷰 탭 클릭 성공")
                time.sleep(2)
            except:
                print(f"[{isbn}] 전체 리뷰 탭 클릭 실패")

            page = 1
            while True:
                print(f" > {page}페이지 수집 중...")
                time.sleep(2)

                review_boxes = driver.find_elements(By.CSS_SELECTOR, "div.reviewInfoGrp")

                for box in review_boxes:
                    try:
                        rating = box.find_element(By.CSS_SELECTOR, "span.total_rating").text.strip()
                    except:
                        rating = "없음"
                    try:
                        nickname = box.find_element(By.CSS_SELECTOR, "em.txt_id").text.strip()
                    except:
                        nickname = "없음"

                    try:
                        more_btn = box.find_element(By.CSS_SELECTOR, "span.review_more")
                        driver.execute_script("arguments[0].click();", more_btn)
                        time.sleep(1)
                    except:
                        pass

                    try:
                        review_div = box.find_element(By.CSS_SELECTOR, "div.review_cont")
                        raw_html = review_div.get_attribute("innerHTML")
                        review_text = review_div.text.strip()
                        if not review_text:
                            soup = BeautifulSoup(raw_html, "html.parser")
                            review_text = soup.get_text(separator=" ", strip=True)
                    except:
                        review_text = "없음"

                    all_reviews.append({
                        'isbn': isbn,
                        'nickname': nickname,
                        'star_rating': rating,
                        'review_content': review_text
                    })

                try:
                    page_btn = driver.find_element(By.XPATH, f'//*[@id="infoset_reviewContentList"]/div[1]/div[1]/div/a[{page+2}]')
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", page_btn)
                    time.sleep(1)
                    driver.execute_script("arguments[0].click();", page_btn)
                    page += 1
                    time.sleep(2)
                except NoSuchElementException:
                    try:
                        next_btn = driver.find_element(By.CSS_SELECTOR, "a.bgYUI.next")
                        if next_btn.is_enabled():
                            next_btn.click()
                            page = 1
                            time.sleep(2)
                        else:
                            break
                    except:
                        break
        except Exception as e:
            print(f"[{isbn}] 수집 실패: {e}")

        # ISBN 200개마다 저장
        if (idx + 1) % 200 == 0:
            df = pd.DataFrame(all_reviews)
            df.to_csv("yes24_reviews.csv", index=False, encoding="utf-8-sig")
            print(f"{idx+1}개 ISBN 처리됨! 임시 저장 완료")

    driver.quit()
    return all_reviews


if __name__ == "__main__":
    # CSV 파일에서 ISBN 리스트 불러오기
    isbn_df = pd.read_csv("yes24_isbn_list.csv", encoding="utf-8-sig")
    isbn_list = isbn_df["ISBN"].astype(str).str.strip().tolist()

    data = crawl_yes24_reviews(isbn_list)
    df = pd.DataFrame(data)
    df.to_csv("yes24_reviews.csv", index=False, encoding="utf-8-sig")
    print("CSV 저장 완료! yes24_reviews.csv")


## 2. 한줄평

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time


def setup_driver():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_experimental_option("detach", True)
    options.add_argument("--disable-blink-features=AutomationControlled")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


def crawl_yes24_oneline_reviews(isbn_list):
    all_reviews = []
    driver = setup_driver()

    for idx, isbn in enumerate(isbn_list):
        print(f"\n [{isbn}] ({idx+1}/{len(isbn_list)}) 수집 시작")
        try:
            driver.get("https://www.yes24.com/")
            time.sleep(2)

            search_box = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "span.schIpt input"))
            )
            search_box.clear()
            search_box.send_keys(isbn)

            search_button = driver.find_element(By.CSS_SELECTOR, "button[title='검색']")
            search_button.click()
            time.sleep(2)

            try:
                detail_link = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "em.img_bdr"))
                )
                detail_link.click()
            except:
                print(f"[{isbn}] 상세페이지 이동 실패")
                continue

            time.sleep(3)
            tab_xpath = '//*[@id="yDetailTabNavWrap"]/div/div[2]/ul/li[2]/a/em[1]'
            try:
                review_tab = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, tab_xpath)))
                driver.execute_script("arguments[0].click();", review_tab)
                time.sleep(2)
            except:
                print(f"[{isbn}] 한줄평 탭 클릭 실패")
                continue

            scroll_xpath = '//*[@id="infoset_rvCmt"]/div[1]'
            for _ in range(30):
                driver.execute_script("window.scrollBy(0, 300);")
                time.sleep(0.5)
                try:
                    driver.find_element(By.XPATH, scroll_xpath)
                    break
                except:
                    continue

            page = 1
            max_pages = 100
            consecutive_failures = 0
            current_page_reviews = set()

            while page <= max_pages:
                print(f" > {page}페이지 수집 중...")
                time.sleep(2)

                try:
                    WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "div.cmtInfoBox"))
                    )
                except TimeoutException:
                    print(f"   페이지 {page} 로딩 실패")
                    consecutive_failures += 1
                    if consecutive_failures >= 3:
                        print("   연속 3회 실패로 중단")
                        break
                    continue

                review_boxes = driver.find_elements(By.CSS_SELECTOR, "div.cmtInfoBox")

                if not review_boxes:
                    print(f"   페이지 {page}에 리뷰가 없음")
                    break

                page_review_contents = []
                for box in review_boxes:
                    try:
                        review_text = box.find_element(By.CSS_SELECTOR, "div.cmt_cont span.txt").text.strip()
                        page_review_contents.append(review_text)
                    except:
                        pass

                if page > 1 and current_page_reviews == set(page_review_contents):
                    print(f"   페이지 {page}: 이전 페이지와 동일한 내용 감지, 크롤링 종료")
                    break

                current_page_reviews = set(page_review_contents)

                page_review_count = 0
                for box in review_boxes:
                    try:
                        rating = box.find_element(By.CSS_SELECTOR, "span.rating").text.strip()
                    except:
                        rating = "없음"

                    try:
                        review = box.find_element(By.CSS_SELECTOR, "div.cmt_cont span.txt").text.strip()
                    except:
                        review = "없음"

                    # 닉네임 수집
                    try:
                        nickname = box.find_element(By.CSS_SELECTOR, "div.cmt_etc em.txt_id a").text.strip()
                    except:
                        try:
                            nickname = box.find_element(By.CSS_SELECTOR, "div.cmt_etc a").text.strip()
                        except:
                            try:
                                nickname = box.find_element(By.CSS_SELECTOR, "div.cmt_etc em.txt_id").text.strip()
                            except:
                                nickname = "익명"

                    all_reviews.append({
                        'isbn': isbn,
                        'nickname': nickname,
                        'star_rating': rating,
                        'review_content': review
                    })
                    page_review_count += 1

                print(f"   페이지 {page}: {page_review_count}개 리뷰 수집")

                try:
                    if page < 10:
                        page_btn_xpath = f'//*[@id="infoset_oneCommentList"]/div[3]/div[1]/div/a[{page+2}]'
                        next_btn = WebDriverWait(driver, 5).until(
                            EC.element_to_be_clickable((By.XPATH, page_btn_xpath)))
                        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_btn)
                        time.sleep(1)
                        driver.execute_script("arguments[0].click();", next_btn)
                    else:
                        next_btn = WebDriverWait(driver, 5).until(
                            EC.element_to_be_clickable((By.CSS_SELECTOR, "a.bgYUI.next")))
                        if next_btn.is_enabled():
                            driver.execute_script("arguments[0].click();", next_btn)
                        else:
                            print("   다음 버튼 비활성화")
                            break
                except Exception as e:
                    print(f"   페이지 이동 실패: {e}")
                    break

                page += 1
                time.sleep(3)

            print(f"[{isbn}] 총 {len([r for r in all_reviews if r['isbn'] == isbn])}개 리뷰 수집 완료")

        except Exception as e:
            print(f"[{isbn}] 수집 실패: {e}")

        if (idx + 1) % 5 == 0:
            df = pd.DataFrame(all_reviews)
            df.to_csv("yes24_oneline_reviews_temp.csv", index=False, encoding="utf-8-sig")
            print(f"{idx+1}개 ISBN 처리됨! 임시 저장 완료 (총 {len(all_reviews)}개 리뷰)")

    driver.quit()
    return all_reviews


if __name__ == "__main__":
    isbn_df = pd.read_csv("비베스트셀러_랜덤.csv", encoding="utf-8-sig")
    isbn_list = isbn_df["ISBN"].astype(str).str.strip().tolist()

    data = crawl_yes24_oneline_reviews(isbn_list)
    df = pd.DataFrame(data)
    df.to_csv("[예사] 한줄평_비베스트셀러.csv", index=False, encoding="utf-8-sig")
    print(f"CSV 저장 완료! 총 {len(data)}개 리뷰 수집됨")



🔥 [9788997700721] (1/9804) 수집 시작🔥
 > 1페이지 수집 중...
   페이지 1 로딩 실패
 > 1페이지 수집 중...
   페이지 1 로딩 실패
 > 1페이지 수집 중...
   페이지 1 로딩 실패
   연속 3회 실패로 중단
📊 [9788997700721] 총 0개 리뷰 수집 완료

🔥 [9788973142644] (2/9804) 수집 시작🔥
 > 1페이지 수집 중...
   페이지 1 로딩 실패
 > 1페이지 수집 중...
   페이지 1 로딩 실패
 > 1페이지 수집 중...
   페이지 1 로딩 실패
   연속 3회 실패로 중단
📊 [9788973142644] 총 0개 리뷰 수집 완료

🔥 [9788988695913] (3/9804) 수집 시작🔥
 > 1페이지 수집 중...
   페이지 1 로딩 실패
 > 1페이지 수집 중...
   페이지 1 로딩 실패
 > 1페이지 수집 중...
   페이지 1 로딩 실패
   연속 3회 실패로 중단
📊 [9788988695913] 총 0개 리뷰 수집 완료

🔥 [9788930010917] (4/9804) 수집 시작🔥
 > 1페이지 수집 중...
   페이지 1 로딩 실패
 > 1페이지 수집 중...
   페이지 1 로딩 실패
 > 1페이지 수집 중...
   페이지 1 로딩 실패
   연속 3회 실패로 중단
📊 [9788930010917] 총 0개 리뷰 수집 완료

🔥 [9788936474614] (5/9804) 수집 시작🔥
 > 1페이지 수집 중...
   페이지 1 로딩 실패
 > 1페이지 수집 중...
   페이지 1 로딩 실패
 > 1페이지 수집 중...
   페이지 1 로딩 실패
   연속 3회 실패로 중단
📊 [9788936474614] 총 0개 리뷰 수집 완료
📅 5개 ISBN 처리됨! 임시 저장 완료 (총 0개 리뷰)

🔥 [9788997091232] (6/9804) 수집 시작🔥
 > 1페이지 수집 중...
   페이지 1 로딩 실패
 > 1페이지 수집 중...
   페